# 🤖 RAG-Based Customer Support Assistant
## Built with LangGraph, ChromaDB, Groq LLM & HITL
---
**Project:** RAG Internship Project
**Knowledge Base:** Structured Segmentation in RAG Systems
**Tools:** LangChain · ChromaDB · LangGraph · Groq · Sentence Transformers

In [7]:
!pip install langchain langchain-community langchain-groq chromadb sentence-transformers langgraph pypdf reportlab -q

print("✅ All libraries installed successfully!")

✅ All libraries installed successfully!


In [4]:
!pip install langchain-huggingface -q
print("✅ Done!")
!pip install langchain-text-splitters -q
print("✅ Done!")

✅ Done!
✅ Done!


## 📦 Step 2: Imports & Configuration
Setting up all libraries and API keys needed for the project.

In [8]:
# ============================================================
# CELL 2: Imports & Configuration (Fixed v3)
# ============================================================

# --- Standard Libraries ---
import os
import warnings
warnings.filterwarnings("ignore")

# --- LangChain Core ---
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# --- Embeddings ---
from langchain_huggingface import HuggingFaceEmbeddings

# --- Vector Store ---
from langchain_community.vectorstores import Chroma

# --- Groq LLM ---
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

# --- LangGraph ---
from langgraph.graph import StateGraph, END
from typing import TypedDict, Optional

# --- API Key ---
os.environ["GROQ_API_KEY"] = "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"

# --- Quick Test ---
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
response = llm.invoke("Say hello in one sentence.")
print("✅ Imports done!")
print("🤖 LLM Test:", response.content)

✅ Imports done!
🤖 LLM Test: Hello, it's nice to meet you and I'm here to help with any questions or topics you'd like to discuss.


## 📄 Step 3: Load PDF & Chunk It
Loading the PDF, splitting it into chunks for embedding.

In [9]:
# ============================================================
# CELL 3: Upload PDF, Load & Chunk It (FIXED VERSION)
# ============================================================

# --- Step 1: Upload PDF ---
from google.colab import files
uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]
print(f"📁 Uploaded file: {pdf_path}")

# --- Step 2: Install required libraries ---
!pip install -q langchain langchain-community langchain-text-splitters pypdf

# --- Step 3: Imports (UPDATED) ---
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- Step 4: Load PDF ---
loader = PyPDFLoader(pdf_path)
pages = loader.load()

print("✅ PDF Loaded!")
print(f"📄 Total pages: {len(pages)}")

# --- Step 5: Chunking ---
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " "]
)

chunks = splitter.split_documents(pages)

print(f"✂️ Total chunks created: {len(chunks)}")

# --- Step 6: Preview ---
print("\n🔍 Preview of first 3 chunks:")
print("=" * 60)

for i, chunk in enumerate(chunks[:3]):
    print(f"\n📌 Chunk {i+1}:")
    print(chunk.page_content[:500])
    print("-" * 40)

Saving structured_segmentation_rag.pdf to structured_segmentation_rag.pdf
📁 Uploaded file: structured_segmentation_rag.pdf
✅ PDF Loaded!
📄 Total pages: 1
✂️ Total chunks created: 4

🔍 Preview of first 3 chunks:

📌 Chunk 1:
Structured Segmentation in RAG Systems
Overview
Structured segmentation is used in Retrieval-Augmented Generation (RAG) systems to divide large
documents into meaningful sections based on context and topic continuity. Each section preserves
a complete idea, ensuring better understanding and retrieval performance.
Section 1: Introduction
RAG systems combine retrieval and generation to produce accurate answers. Organizing
----------------------------------------

📌 Chunk 2:
documents into logical sections helps the system focus only on relevant information instead of
processing the entire text.
Section 2: Preprocessing
Text is cleaned by removing noise, unnecessary symbols, and formatting issues. This step ensures
that only meaningful and usable content is passed to th

## 🧠 Step 4: Embed Chunks & Store in ChromaDB
Converting text chunks into vector embeddings and storing them in ChromaDB for retrieval.

In [10]:
# ============================================================
# CELL 4: Embed Chunks & Store in ChromaDB
# ============================================================

# --- Step 1: Load Embedding Model ---
print("⏳ Loading embedding model (first time may take 1-2 mins)...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("✅ Embedding model loaded!")

# --- Step 2: Store in ChromaDB ---
print("\n⏳ Storing chunks in ChromaDB...")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="/content/drive/My Drive/chroma_db"
)
print("✅ ChromaDB created and saved to Google Drive!")

# --- Step 3: Test Retrieval ---
print("\n🔍 Testing retrieval with sample query...")
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

test_query = "What is AI?"
results = retriever.invoke(test_query)

print(f"\n📌 Query: '{test_query}'")
print(f"📦 Retrieved {len(results)} chunks:")
print("=" * 60)
for i, doc in enumerate(results):
    print(f"\n📄 Result {i+1}:")
    print(doc.page_content)
    print("-" * 40)

⏳ Loading embedding model (first time may take 1-2 mins)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded!

⏳ Storing chunks in ChromaDB...
✅ ChromaDB created and saved to Google Drive!

🔍 Testing retrieval with sample query...

📌 Query: 'What is AI?'
📦 Retrieved 3 chunks:

📄 Result 1:
Conclusion
By organizing information into meaningful sections, RAG systems achieve higher accuracy, better
context preservation, and more relevant responses, making them highly effective for modern AI
applications.
----------------------------------------

📄 Result 2:
documents into logical sections helps the system focus only on relevant information instead of
processing the entire text.
Section 2: Preprocessing
Text is cleaned by removing noise, unnecessary symbols, and formatting issues. This step ensures
that only meaningful and usable content is passed to the next stage.
Section 3: Sentence-Level Analysis
The document is divided into sentences, and relationships between them are analyzed using
----------------------------------------

📄 Result 3:
embeddings to understand semanti

## 🔗 Step 5: Build LangGraph Workflow
Building the graph-based workflow with nodes, edges, state, conditional routing and HITL escalation.

In [11]:
# ============================================================
# CELL 5: LangGraph Workflow with RAG + Routing + HITL
# ============================================================

# --- Step 1: Define State ---
class GraphState(TypedDict):
    question: str                  # User's question
    retrieved_chunks: list         # Chunks from ChromaDB
    answer: str                    # LLM generated answer
    confidence: str                # high / low
    escalated: bool                # Was it escalated to human?
    human_response: Optional[str]  # Human agent's response

# ============================================================
# --- Step 2: Define Nodes ---
# ============================================================

# NODE 1: Retrieve relevant chunks from ChromaDB
def retrieve_node(state: GraphState) -> GraphState:
    print("🔍 [Node 1] Retrieving relevant chunks...")
    question = state["question"]
    docs = retriever.invoke(question)
    state["retrieved_chunks"] = [doc.page_content for doc in docs]
    print(f"   ✅ Retrieved {len(docs)} chunks")
    return state

# NODE 2: Generate answer using LLM
def generate_node(state: GraphState) -> GraphState:
    print("🤖 [Node 2] Generating answer with LLM...")
    question = state["question"]
    context = "\n\n".join(state["retrieved_chunks"])

    prompt = ChatPromptTemplate.from_template("""
You are a helpful customer support assistant.
Use ONLY the context below to answer the question.
If the answer is not in the context, say "I don't have enough information."

Context:
{context}

Question: {question}

Answer clearly and concisely.
""")

    chain = prompt | llm
    response = chain.invoke({"context": context, "question": question})
    state["answer"] = response.content
    print(f"   ✅ Answer generated")
    return state

# NODE 3: Check confidence / intent routing
def confidence_node(state: GraphState) -> GraphState:
    print("🧭 [Node 3] Checking confidence & routing...")
    answer = state["answer"].lower()

    # Low confidence triggers
    low_confidence_phrases = [
        "i don't have enough information",
        "i don't know",
        "not mentioned",
        "unclear",
        "cannot find",
        "no information"
    ]

    if any(phrase in answer for phrase in low_confidence_phrases):
        state["confidence"] = "low"
        print("   ⚠️  Low confidence detected → will escalate")
    else:
        state["confidence"] = "high"
        print("   ✅ High confidence → will answer directly")

    state["escalated"] = False
    return state

# NODE 4: HITL Escalation — human takes over
def hitl_node(state: GraphState) -> GraphState:
    print("\n🚨 [Node 4 - HITL] Escalating to human agent...")
    print("=" * 60)
    print(f"❓ User Question: {state['question']}")
    print(f"🤖 Bot could not answer confidently.")
    print("=" * 60)

    # Simulate human agent response
    human_input = input("👨‍💼 Human Agent — Type your response: ")
    state["human_response"] = human_input
    state["escalated"] = True
    print("   ✅ Human response recorded")
    return state

# NODE 5: Final output
def output_node(state: GraphState) -> GraphState:
    print("\n📤 [Node 5] Preparing final output...")
    return state

# ============================================================
# --- Step 3: Routing Logic ---
# ============================================================

def route_after_confidence(state: GraphState) -> str:
    if state["confidence"] == "low":
        return "hitl"       # Go to human
    else:
        return "output"     # Go to final output

# ============================================================
# --- Step 4: Build the Graph ---
# ============================================================

workflow = StateGraph(GraphState)

# Add nodes
workflow.add_node("retrieve",   retrieve_node)
workflow.add_node("generate",   generate_node)
workflow.add_node("confidence", confidence_node)
workflow.add_node("hitl",       hitl_node)
workflow.add_node("output",     output_node)

# Add edges
workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve",   "generate")
workflow.add_edge("generate",   "confidence")
workflow.add_conditional_edges(
    "confidence",
    route_after_confidence,
    {
        "hitl":   "hitl",
        "output": "output"
    }
)
workflow.add_edge("hitl",   "output")
workflow.add_edge("output", END)

# Compile
app = workflow.compile()
print("✅ LangGraph workflow compiled successfully!")
print("\n📊 Graph Structure:")
print("   retrieve → generate → confidence")
print("                              ↓")
print("                    [high] output → END")
print("                    [low]  hitl → output → END")

✅ LangGraph workflow compiled successfully!

📊 Graph Structure:
   retrieve → generate → confidence
                              ↓
                    [high] output → END
                    [low]  hitl → output → END


## 🧪 Step 6: Full End-to-End Test
Testing the complete RAG pipeline with sample queries including HITL escalation.

In [16]:
# ============================================================
# CELL 6: Full End-to-End Test (MULTIPLE QUESTIONS)
# ============================================================

def run_bot(question: str):
    print("\n" + "=" * 60)
    print(f"💬 USER QUESTION: {question}")
    print("=" * 60)

    # Initial state
    initial_state = {
        "question": question,
        "retrieved_chunks": [],
        "answer": "",
        "confidence": "",
        "escalated": False,
        "human_response": None
    }

    # Run the graph
    final_state = app.invoke(initial_state)

    # Display final answer
    print("\n" + "=" * 60)
    print("📋 FINAL RESPONSE:")
    print("=" * 60)

    if final_state["escalated"]:
        print(f"🚨 Status    : Escalated to Human Agent")
        print(f"👨‍💼 Response  : {final_state['human_response']}")
    else:
        print(f"✅ Status    : Answered by Bot")
        print(f"🤖 Answer    : {final_state['answer']}")

    print("=" * 60)


# ============================================================
# ✅ TEST CASES (Based on YOUR PDF)
# ============================================================

# 🔹 TEST 1: Basic Concept (HIGH CONFIDENCE)
run_bot("What is RAG?")

# 🔹 TEST 2: Section-specific question
run_bot("What happens in preprocessing stage?")

# 🔹 TEST 3: Deep understanding
run_bot("How does sentence-level analysis work in RAG systems?")

# 🔹 TEST 4: Context grouping concept
run_bot("What is contextual grouping and why is it important?")

# 🔹 TEST 8: OUT-OF-CONTEXT (Should trigger HITL 🚨)
run_bot("Who is Virat Kohli?")

# 🔹 TEST 9: COMPLETELY IRRELEVANT
run_bot("What is quantum physics?")


💬 USER QUESTION: What is RAG?
🔍 [Node 1] Retrieving relevant chunks...
   ✅ Retrieved 3 chunks
🤖 [Node 2] Generating answer with LLM...
   ✅ Answer generated
🧭 [Node 3] Checking confidence & routing...
   ✅ High confidence → will answer directly

📤 [Node 5] Preparing final output...

📋 FINAL RESPONSE:
✅ Status    : Answered by Bot
🤖 Answer    : RAG stands for Retrieval-Augmented Generation.

💬 USER QUESTION: What happens in preprocessing stage?
🔍 [Node 1] Retrieving relevant chunks...
   ✅ Retrieved 3 chunks
🤖 [Node 2] Generating answer with LLM...
   ✅ Answer generated
🧭 [Node 3] Checking confidence & routing...
   ✅ High confidence → will answer directly

📤 [Node 5] Preparing final output...

📋 FINAL RESPONSE:
✅ Status    : Answered by Bot
🤖 Answer    : In the preprocessing stage, text is cleaned by removing noise, unnecessary symbols, and formatting issues to ensure only meaningful and usable content is passed to the next stage.

💬 USER QUESTION: How does sentence-level analysis wo

In [17]:
# ============================================================
# CELL 7: Test HITL Escalation
# ============================================================

# TEST 2: This will trigger HITL (question not in PDF)
run_bot("How does sentence-level analysis work in RAG systems?")


💬 USER QUESTION: How does sentence-level analysis work in RAG systems?
🔍 [Node 1] Retrieving relevant chunks...
   ✅ Retrieved 3 chunks
🤖 [Node 2] Generating answer with LLM...
   ✅ Answer generated
🧭 [Node 3] Checking confidence & routing...
   ✅ High confidence → will answer directly

📤 [Node 5] Preparing final output...

📋 FINAL RESPONSE:
✅ Status    : Answered by Bot
🤖 Answer    : In RAG systems, sentence-level analysis works by dividing the document into sentences and analyzing the relationships between them.


In [18]:
# ============================================================
# CELL 8: Final Summary Test — Both Paths
# ============================================================

print("🧪 RUNNING FULL SYSTEM TEST")
print("=" * 60)

# 🔹 TEST 5: Storage question
run_bot("Where are the sections stored in RAG systems?")

# 🔹 TEST 6: Purpose-based question
run_bot("Why is structured segmentation important in RAG?")

# 🔹 TEST 7: Conclusion-based question
run_bot("What are the benefits of structured segmentation?")

🧪 RUNNING FULL SYSTEM TEST

💬 USER QUESTION: Where are the sections stored in RAG systems?
🔍 [Node 1] Retrieving relevant chunks...
   ✅ Retrieved 3 chunks
🤖 [Node 2] Generating answer with LLM...
   ✅ Answer generated
🧭 [Node 3] Checking confidence & routing...
   ✅ High confidence → will answer directly

📤 [Node 5] Preparing final output...

📋 FINAL RESPONSE:
✅ Status    : Answered by Bot
🤖 Answer    : The sections are stored in a vector database, such as FAISS or Pinecone.

💬 USER QUESTION: Why is structured segmentation important in RAG?
🔍 [Node 1] Retrieving relevant chunks...
   ✅ Retrieved 3 chunks
🤖 [Node 2] Generating answer with LLM...
   ✅ Answer generated
🧭 [Node 3] Checking confidence & routing...
   ✅ High confidence → will answer directly

📤 [Node 5] Preparing final output...

📋 FINAL RESPONSE:
✅ Status    : Answered by Bot
🤖 Answer    : Structured segmentation is important in RAG systems because it allows them to divide large documents into meaningful sections, preservi

**📌 Conclusion**

This project implements a RAG-based Customer Support Assistant that retrieves relevant information from a PDF and generates accurate, context-aware responses. Using a LangGraph workflow, it enables conditional routing and supports Human-in-the-Loop (HITL) for handling low-confidence or out-of-domain queries.

The system improves response reliability, prevents incorrect answers, and demonstrates a scalable approach for intelligent customer support applications.